# 12 GHSL Six-City Urban Growth Model Training

This notebook continues from the validated inventory produced by Notebook 11.

**Training cities:** Bengaluru, Hyderabad, Pune Metropolitan Region, Chennai Metropolitan Region, Mumbai Metropolitan Region, and Delhi NCR.  
**Held-out city:** Jaipur. Jaipur is not read or used here.

The notebook uses `/content/drive/MyDrive/remote_exec.sh` as the single source of truth for EC2 connection details. All Spark and HDFS work runs remotely on EC2. Colab only submits commands and stores compact reports.

## Outputs

- Harmonized longitudinal data: `/data/ghsl/modeling/six_city_longitudinal_v1`
- Supervised transitions: `/data/ghsl/modeling/six_city_transitions_v1`
- Spark ML models: `/data/ghsl/modeling/models/six_city_growth_v1`
- Metrics and manifest: Google Drive under `GHSL_Urban_Growth_Model/model_training_v1`

Mounts Google Drive and prepares the temporary Colab SSH key.

Uses remote_exec.sh for every EC2 command.

Loads and validates six_city_inventory.json.

Runs the HDFS, Spark, and Sedona preflight.

Discovers the exact built-surface target column rather than silently guessing.

Creates the harmonized 294,000-row longitudinal dataset.

Creates 264,600 five-year transition examples.

Uses chronological splits:

Training: transitions ending by 2010

Validation: 2010 → 2015

Test: 2015 → 2020

Evaluates a persistence baseline.

Trains:

Spark ML Linear Regression

Spark ML Random Forest Regression

Reports MAE, RMSE, and R2R^2R2.

Reports separate test metrics for each city.

Saves models in HDFS and a training manifest in Google Drive.

Keeps Jaipur completely outside training.

In [1]:
# ============================================================
# CELL 1: DRIVE, REMOTE EXECUTOR, CONSTANTS
# ============================================================
from google.colab import drive
from pathlib import Path
import base64
import json
import os
import shutil
import subprocess
import time

drive.mount("/content/drive", force_remount=False)

REMOTE_EXEC = Path("/content/drive/MyDrive/remote_exec.sh")
DRIVE_SSH_KEY = Path("/content/drive/MyDrive/hadoop-hdfs.pem")
COLAB_SSH_KEY = Path("/content/hadoop-hdfs.pem")
ARTIFACT_ROOT = Path("/content/drive/MyDrive/GHSL_Urban_Growth_Model/model_training_v1")
INVENTORY_PATH = Path("/content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_inventory.json")

if not REMOTE_EXEC.is_file():
    raise FileNotFoundError(f"Missing remote executor: {REMOTE_EXEC}")
if not DRIVE_SSH_KEY.is_file():
    raise FileNotFoundError(f"Missing SSH key: {DRIVE_SSH_KEY}")
if not INVENTORY_PATH.is_file():
    raise FileNotFoundError(f"Missing Notebook 11 inventory: {INVENTORY_PATH}")

REMOTE_EXEC.chmod(0o700)
if COLAB_SSH_KEY.exists():
    COLAB_SSH_KEY.unlink()
shutil.copy2(DRIVE_SSH_KEY, COLAB_SSH_KEY)
COLAB_SSH_KEY.chmod(0o600)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_CITIES = 6
EXPECTED_EPOCHS = [1975,1980,1985,1990,1995,2000,2005,2010,2015,2020]
EXPECTED_LONGITUDINAL_ROWS = 294000
EXPECTED_TRANSITION_ROWS = 264600
LONGITUDINAL_OUTPUT = "/data/ghsl/modeling/six_city_longitudinal_v1"
TRANSITION_OUTPUT = "/data/ghsl/modeling/six_city_transitions_v1"
MODEL_ROOT = "/data/ghsl/modeling/models/six_city_growth_v1"

inventory = json.loads(INVENTORY_PATH.read_text(encoding="utf-8"))
if inventory.get("expected_total_rows") != EXPECTED_LONGITUDINAL_ROWS:
    raise RuntimeError("Notebook 11 inventory row count is not 294000.")
if len(inventory.get("training_city_slugs", [])) != EXPECTED_CITIES:
    raise RuntimeError("Notebook 11 inventory does not contain six training cities.")

print("Drive mounted: PASS")
print("remote_exec.sh:", REMOTE_EXEC)
print("Notebook 11 inventory:", INVENTORY_PATH)
print("Artifact root:", ARTIFACT_ROOT)
print("Jaipur included in training: NO")
print("CELL 1: SUCCESS")

Mounted at /content/drive
Drive mounted: PASS
remote_exec.sh: /content/drive/MyDrive/remote_exec.sh
Notebook 11 inventory: /content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_inventory.json
Artifact root: /content/drive/MyDrive/GHSL_Urban_Growth_Model/model_training_v1
Jaipur included in training: NO
CELL 1: SUCCESS


In [2]:
# ============================================================
# CELL 2: REMOTE HELPER AND CONNECTION TEST
# ============================================================
def run_remote(command, timeout=300, show_output=True):
    result = subprocess.run(
        [str(REMOTE_EXEC), command],
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    if show_output and result.stdout:
        print(result.stdout)
    if show_output and result.stderr:
        print("REMOTE STDERR:")
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            "Remote command failed with return code " + str(result.returncode)
        )
    return result

r = run_remote(r'''set -e
echo "REMOTE_EXEC_CONNECTION_SUCCESS"
echo "Remote user: $(whoami)"
echo "Remote host: $(hostname)"
''', timeout=90)
if "REMOTE_EXEC_CONNECTION_SUCCESS" not in r.stdout:
    raise RuntimeError("Remote connection marker missing.")
print("CELL 2: SUCCESS")

REMOTE_EXEC_CONNECTION_SUCCESS
Remote user: ubuntu
Remote host: ip-172-31-30-94

REMOTE STDERR:

CELL 2: SUCCESS


In [3]:
# ============================================================
# CELL 3: HDFS/SPARK PREFLIGHT
# ============================================================
r = run_remote(r'''set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
for daemon in NameNode DataNode SecondaryNameNode; do
  jps | awk '{print $2}' | grep -qx "$daemon"
done
hdfs dfsadmin -safemode get | grep -qi "Safe mode is OFF"
command -v spark-submit >/dev/null
hdfs dfs -test -d /data/ghsl/curated/cities
python3 - <<'PY_REMOTE'
import importlib.util
missing=[x for x in ["pyspark","sedona"] if importlib.util.find_spec(x) is None]
if missing: raise RuntimeError("Missing packages: "+repr(missing))
print("PySpark/Sedona imports: PASS")
PY_REMOTE
echo "MODEL_TRAINING_PREFLIGHT_SUCCESS"
''', timeout=300)
if "MODEL_TRAINING_PREFLIGHT_SUCCESS" not in r.stdout:
    raise RuntimeError("Preflight marker missing.")
print("CELL 3: SUCCESS")

PySpark/Sedona imports: PASS
MODEL_TRAINING_PREFLIGHT_SUCCESS

REMOTE STDERR:

CELL 3: SUCCESS


In [4]:
# ============================================================
# CELL 4: BUILD EXACT 60-PARTITION PAYLOAD FROM FROZEN INVENTORY
# ============================================================
training_cities = inventory["training_cities"]
records = []
for city_name, config in training_cities.items():
    for epoch in EXPECTED_EPOCHS:
        records.append({
            "city_name": city_name,
            "city_slug": config["city_slug"],
            "epoch": epoch,
            "path": config["enriched_root"] + "/epoch=" + str(epoch),
        })
if len(records) != 60:
    raise RuntimeError(f"Expected 60 partitions, found {len(records)}")
if len({(x["city_slug"], x["epoch"]) for x in records}) != 60:
    raise RuntimeError("Duplicate city-epoch records generated.")
PAYLOAD_B64 = base64.b64encode(json.dumps(records, sort_keys=True).encode()).decode()
print("Training cities:", sorted({x["city_slug"] for x in records}))
print("Partition records: 60")
print("CELL 4: SUCCESS")

Training cities: ['bengaluru', 'chennai_metro', 'delhi_ncr', 'hyderabad', 'mumbai_metro', 'pune_metro']
Partition records: 60
CELL 4: SUCCESS


In [5]:
# ============================================================
# CELL 5: DISCOVER MODEL COLUMNS AND FREEZE THE TARGET
# Read-only Spark job. Fails rather than guessing a wrong target.
# ============================================================
cmd = r'''set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
SCRIPT=/tmp/ghsl_model_schema_probe.py
cat > "$SCRIPT" <<'PY_REMOTE'
from pyspark.sql import SparkSession
import json
spark=SparkSession.builder.appName("GHSL-Model-Schema-Probe").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
try:
    path="hdfs://localhost:9000/data/ghsl/curated/cities/city=bengaluru/enriched_grid_year/epoch=2020"
    df=spark.read.parquet(path)
    fields=[{"name":f.name,"type":f.dataType.simpleString()} for f in df.schema.fields]
    print("SCHEMA_JSON_BEGIN")
    print(json.dumps(fields, sort_keys=True))
    print("SCHEMA_JSON_END")
finally:
    spark.stop()
PY_REMOTE
spark-submit --master local[1] --driver-memory 3g --conf spark.ui.enabled=false "$SCRIPT"
rm -f "$SCRIPT"
'''
r=run_remote(cmd, timeout=900, show_output=False)
b="SCHEMA_JSON_BEGIN"; e="SCHEMA_JSON_END"
if b not in r.stdout or e not in r.stdout:
    print(r.stdout[-8000:]); print(r.stderr[-4000:]); raise RuntimeError("Schema markers missing")
fields=json.loads(r.stdout.split(b,1)[1].split(e,1)[0].strip())
FIELD_TYPES={x["name"]:x["type"] for x in fields}
print("Columns discovered:", len(FIELD_TYPES))
for name in sorted(FIELD_TYPES):
    print(" -", name, ":", FIELD_TYPES[name])

TARGET_CANDIDATES=[
    "built_surface_m2",
    "built_surface_total_m2",
    "built_surface_sum_m2",
    "built_surface",
    "built_s_m2",
]
TARGET_SOURCE_COLUMNS=[x for x in TARGET_CANDIDATES if x in FIELD_TYPES]
if len(TARGET_SOURCE_COLUMNS) != 1:
    raise RuntimeError(
        "Expected exactly one supported built-surface target column. "
        f"Candidates found={TARGET_SOURCE_COLUMNS}. Inspect the printed schema and update TARGET_CANDIDATES only."
    )
TARGET_SOURCE_COLUMN=TARGET_SOURCE_COLUMNS[0]
print("Selected current-state target source:", TARGET_SOURCE_COLUMN)
print("Training target: next_"+TARGET_SOURCE_COLUMN)
print("CELL 5: SUCCESS")

Columns discovered: 48
 - built_100m_pixels : bigint
 - built_coverage_fraction : decimal(27,6)
 - built_observed_grid_area_m2 : decimal(27,1)
 - built_quality_flag : string
 - built_surface_fraction : double
 - built_surface_nres_fraction : double
 - built_surface_nres_m2 : double
 - built_surface_percent : double
 - built_surface_residential_fraction : double
 - built_surface_residential_m2 : double
 - built_surface_source_path : string
 - built_surface_total_m2 : double
 - built_valid_100m_pixels : bigint
 - city : string
 - epoch : int
 - geometry_crs : string
 - geometry_wkb : binary
 - grid_1km_col : bigint
 - grid_1km_row : bigint
 - grid_cell_id : string
 - grid_max_x : double
 - grid_max_y : double
 - grid_min_x : double
 - grid_min_y : double
 - grid_resolution_m : double
 - is_complete_all_products : boolean
 - is_complete_built_cell : boolean
 - is_complete_population_cell : boolean
 - is_urban : boolean
 - is_urban_centre : boolean
 - nres_100m_pixels : bigint
 - nres_shar

In [6]:
# ============================================================
# CELL 6: BUILD HARMONIZED 294,000-ROW LONGITUDINAL DATASET
# Safe behavior: refuses to overwrite an existing destination.
# ============================================================
payload = {
    "records": records,
    "output": LONGITUDINAL_OUTPUT,
    "target_source_column": TARGET_SOURCE_COLUMN,
}
payload_b64 = base64.b64encode(json.dumps(payload, sort_keys=True).encode()).decode()
cmd = r'''set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
PAYLOAD=/tmp/ghsl_longitudinal_payload.json
SCRIPT=/tmp/ghsl_build_longitudinal.py
echo "__PAYLOAD__" | base64 -d > "$PAYLOAD"
cat > "$SCRIPT" <<'PY_REMOTE'
from pyspark.sql import SparkSession, functions as F
from functools import reduce
import json, sys
p=json.load(open(sys.argv[1]))
records=p["records"]; out=p["output"]
spark=SparkSession.builder.appName("GHSL-Build-Six-City-Longitudinal").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
try:
    jvm=spark._jvm
    conf=spark._jsc.hadoopConfiguration()
    fs=jvm.org.apache.hadoop.fs.FileSystem.get(conf)
    out_path=jvm.org.apache.hadoop.fs.Path(out)
    if fs.exists(out_path):
        raise RuntimeError("Refusing to overwrite existing output: "+out)
    frames=[]
    for rec in records:
        df=spark.read.parquet(rec["path"])
        for c in ["built_observed_grid_area_m2","built_coverage_fraction"]:
            if c in df.columns: df=df.withColumn(c,F.col(c).cast("double"))
        if "epoch" in df.columns: df=df.drop("epoch")
        if "city_slug" in df.columns: df=df.drop("city_slug")
        if "city_name" in df.columns: df=df.drop("city_name")
        df=(df.withColumn("city_name",F.lit(rec["city_name"]))
              .withColumn("city_slug",F.lit(rec["city_slug"]))
              .withColumn("epoch",F.lit(int(rec["epoch"])).cast("int")))
        frames.append(df)
    combined=reduce(lambda a,b:a.unionByName(b,allowMissingColumns=False),frames)
    rows=combined.count()
    keys=combined.select("city_slug","epoch","grid_cell_id").distinct().count()
    if rows!=294000 or keys!=294000:
        raise RuntimeError(f"Pre-write validation failed rows={rows} keys={keys}")
    city_counts={r["city_slug"]:r["count"] for r in combined.groupBy("city_slug").count().collect()}
    if len(city_counts)!=6 or any(v!=49000 for v in city_counts.values()):
        raise RuntimeError("Invalid city counts: "+repr(city_counts))
    (combined.repartition("city_slug","epoch")
      .write.mode("errorifexists").partitionBy("city_slug","epoch").parquet(out))
    check=spark.read.parquet(out)
    post_rows=check.count(); post_keys=check.select("city_slug","epoch","grid_cell_id").distinct().count()
    if post_rows!=294000 or post_keys!=294000:
        raise RuntimeError(f"Post-write validation failed rows={post_rows} keys={post_keys}")
    result={"output":out,"rows":post_rows,"keys":post_keys,"city_counts":city_counts,"columns":check.columns}
    print("LONGITUDINAL_JSON_BEGIN"); print(json.dumps(result,sort_keys=True)); print("LONGITUDINAL_JSON_END")
    print("SIX_CITY_LONGITUDINAL_SUCCESS")
finally:
    spark.stop()
PY_REMOTE
spark-submit --master local[2] --driver-memory 5g --conf spark.ui.enabled=false --conf spark.sql.shuffle.partitions=24 --conf spark.driver.maxResultSize=1g "$SCRIPT" "$PAYLOAD"
rm -f "$SCRIPT" "$PAYLOAD"
'''.replace("__PAYLOAD__", payload_b64)
r=run_remote(cmd, timeout=7200, show_output=True)
if "SIX_CITY_LONGITUDINAL_SUCCESS" not in r.stdout: raise RuntimeError("Longitudinal success marker missing")
print("CELL 6: SUCCESS")

LONGITUDINAL_JSON_BEGIN
{"city_counts": {"bengaluru": 49000, "chennai_metro": 49000, "delhi_ncr": 49000, "hyderabad": 49000, "mumbai_metro": 49000, "pune_metro": 49000}, "columns": ["city", "grid_cell_id", "grid_1km_col", "grid_1km_row", "grid_min_x", "grid_max_x", "grid_min_y", "grid_max_y", "valid_100m_pixels", "population_coverage_fraction", "population_total", "population_mean_100m", "population_min_100m", "population_max_100m", "zero_population_pixels", "positive_population_pixels", "smod_class", "settlement_group", "is_urban", "is_urban_centre", "built_valid_100m_pixels", "built_observed_grid_area_m2", "built_coverage_fraction", "built_surface_total_m2", "built_surface_nres_m2", "built_surface_residential_m2", "built_surface_fraction", "built_surface_percent", "built_surface_nres_fraction", "built_surface_residential_fraction", "nres_share_of_built_surface", "built_100m_pixels", "nres_100m_pixels", "is_complete_built_cell", "built_quality_flag", "population_source_path", "smod_so

In [7]:
# ============================================================
# CELL 7: BUILD 264,600 SUPERVISED FIVE-YEAR TRANSITIONS
# Target = next epoch built surface; growth is retained as analysis field.
# ============================================================
payload={"input":LONGITUDINAL_OUTPUT,"output":TRANSITION_OUTPUT,"target":TARGET_SOURCE_COLUMN}
payload_b64=base64.b64encode(json.dumps(payload).encode()).decode()
cmd=r'''set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
PAYLOAD=/tmp/ghsl_transition_payload.json
SCRIPT=/tmp/ghsl_build_transitions.py
echo "__PAYLOAD__" | base64 -d > "$PAYLOAD"
cat > "$SCRIPT" <<'PY_REMOTE'
from pyspark.sql import SparkSession, functions as F, Window
import json, sys
p=json.load(open(sys.argv[1])); inp=p["input"]; out=p["output"]; target=p["target"]
spark=SparkSession.builder.appName("GHSL-Build-Transitions").getOrCreate(); spark.sparkContext.setLogLevel("WARN")
try:
    fs=spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
    if fs.exists(spark._jvm.org.apache.hadoop.fs.Path(out)): raise RuntimeError("Refusing to overwrite existing output: "+out)
    df=spark.read.parquet(inp)
    if target not in df.columns: raise RuntimeError("Target source missing: "+target)
    w=Window.partitionBy("city_slug","grid_cell_id").orderBy("epoch")
    t=(df.withColumn("to_epoch",F.lead("epoch").over(w))
         .withColumn("label",F.lead(F.col(target).cast("double")).over(w))
         .withColumn("current_built_surface_m2",F.col(target).cast("double"))
         .withColumn("built_surface_growth_m2",F.col("label")-F.col("current_built_surface_m2"))
         .withColumnRenamed("epoch","from_epoch")
         .filter((F.col("to_epoch").isNotNull()) & ((F.col("to_epoch")-F.col("from_epoch"))==5)))
    rows=t.count(); keys=t.select("city_slug","grid_cell_id","from_epoch","to_epoch").distinct().count()
    if rows!=264600 or keys!=264600: raise RuntimeError(f"Transition validation failed rows={rows} keys={keys}")
    split_counts={r["to_epoch"]:r["count"] for r in t.groupBy("to_epoch").count().collect()}
    if len(split_counts)!=9 or any(v!=29400 for v in split_counts.values()): raise RuntimeError("Invalid transition epoch counts: "+repr(split_counts))
    (t.repartition("to_epoch","city_slug").write.mode("errorifexists").partitionBy("to_epoch","city_slug").parquet(out))
    check=spark.read.parquet(out)
    if check.count()!=264600: raise RuntimeError("Post-write transition count failed")
    result={"output":out,"rows":rows,"keys":keys,"split_counts":split_counts,"columns":check.columns}
    print("TRANSITION_JSON_BEGIN"); print(json.dumps(result,sort_keys=True)); print("TRANSITION_JSON_END")
    print("SIX_CITY_TRANSITIONS_SUCCESS")
finally:
    spark.stop()
PY_REMOTE
spark-submit --master local[2] --driver-memory 5g --conf spark.ui.enabled=false --conf spark.sql.shuffle.partitions=24 "$SCRIPT" "$PAYLOAD"
rm -f "$SCRIPT" "$PAYLOAD"
'''.replace("__PAYLOAD__",payload_b64)
r=run_remote(cmd,timeout=7200,show_output=True)
if "SIX_CITY_TRANSITIONS_SUCCESS" not in r.stdout: raise RuntimeError("Transition success marker missing")
print("CELL 7: SUCCESS")

TRANSITION_JSON_BEGIN
{"columns": ["city", "grid_cell_id", "grid_1km_col", "grid_1km_row", "grid_min_x", "grid_max_x", "grid_min_y", "grid_max_y", "valid_100m_pixels", "population_coverage_fraction", "population_total", "population_mean_100m", "population_min_100m", "population_max_100m", "zero_population_pixels", "positive_population_pixels", "smod_class", "settlement_group", "is_urban", "is_urban_centre", "built_valid_100m_pixels", "built_observed_grid_area_m2", "built_coverage_fraction", "built_surface_total_m2", "built_surface_nres_m2", "built_surface_residential_m2", "built_surface_fraction", "built_surface_percent", "built_surface_nres_fraction", "built_surface_residential_fraction", "nres_share_of_built_surface", "built_100m_pixels", "nres_100m_pixels", "is_complete_built_cell", "built_quality_flag", "population_source_path", "smod_source_path", "built_surface_source_path", "population_quality_flag", "is_complete_population_cell", "is_complete_all_products", "population_per_buil

In [8]:
# ============================================================
# CELL 8: TRAIN BASELINES, LINEAR REGRESSION, RANDOM FOREST
# Chronological split:
#   Train: to_epoch <= 2010
#   Validation: to_epoch == 2015
#   Test: to_epoch == 2020
# ============================================================
payload={"input":TRANSITION_OUTPUT,"model_root":MODEL_ROOT,"target_source":TARGET_SOURCE_COLUMN}
payload_b64=base64.b64encode(json.dumps(payload).encode()).decode()
cmd=r'''set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
PAYLOAD=/tmp/ghsl_train_payload.json
SCRIPT=/tmp/ghsl_train_models.py
echo "__PAYLOAD__" | base64 -d > "$PAYLOAD"
cat > "$SCRIPT" <<'PY_REMOTE'
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import NumericType
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, Imputer
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
import json, math, sys
p=json.load(open(sys.argv[1])); inp=p["input"]; root=p["model_root"]; target_source=p["target_source"]
spark=SparkSession.builder.appName("GHSL-Train-Urban-Growth-Models").getOrCreate(); spark.sparkContext.setLogLevel("WARN")
try:
    fs=spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
    if fs.exists(spark._jvm.org.apache.hadoop.fs.Path(root)): raise RuntimeError("Refusing to overwrite existing model root: "+root)
    df=spark.read.parquet(inp).filter(F.col("label").isNotNull() & (~F.isnan("label")))
    excluded={
      "label","built_surface_growth_m2","to_epoch","city_name","city_slug","grid_cell_id",
      "geometry","geom","wkt","current_built_surface_m2", target_source
    }
    # Also block obvious future/target/leakage columns.
    def leakage(name):
        n=name.lower()
        return n.startswith("next_") or n.startswith("target_") or "growth_1975_2020" in n
    numeric=[f.name for f in df.schema.fields if isinstance(f.dataType,NumericType)]
    feature_cols=[c for c in numeric if c not in excluded and not leakage(c)]
    # Keep from_epoch as a legitimate temporal predictor.
    required=["from_epoch","built_coverage_fraction","population_coverage_fraction"]
    missing=[c for c in required if c not in feature_cols]
    if missing: raise RuntimeError("Required features missing: "+repr(missing))
    if len(feature_cols)<5: raise RuntimeError("Too few numeric features: "+repr(feature_cols))
    train=df.filter(F.col("to_epoch")<=2010)
    valid=df.filter(F.col("to_epoch")==2015)
    test=df.filter(F.col("to_epoch")==2020)
    counts={"train":train.count(),"validation":valid.count(),"test":test.count()}
    if counts!={"train":205800,"validation":29400,"test":29400}: raise RuntimeError("Unexpected split counts: "+repr(counts))
    def metrics(pred):
        return {
          "mae": RegressionEvaluator(labelCol="label",predictionCol="prediction",metricName="mae").evaluate(pred),
          "rmse": RegressionEvaluator(labelCol="label",predictionCol="prediction",metricName="rmse").evaluate(pred),
          "r2": RegressionEvaluator(labelCol="label",predictionCol="prediction",metricName="r2").evaluate(pred),
        }
    results={"features":feature_cols,"split_counts":counts,"models":{}}
    # Persistence baseline: next built surface equals current built surface.
    for split_name,split in [("validation",valid),("test",test)]:
        pred=split.withColumn("prediction",F.col("current_built_surface_m2"))
        results["models"].setdefault("persistence",{})[split_name]=metrics(pred)
    imputed=["imp_"+c for c in feature_cols]
    pre=[Imputer(inputCols=feature_cols,outputCols=imputed,strategy="median"),VectorAssembler(inputCols=imputed,outputCol="features",handleInvalid="keep")]
    specs={
      "linear_regression": LinearRegression(featuresCol="features",labelCol="label",predictionCol="prediction",maxIter=100,regParam=0.01,elasticNetParam=0.0),
      "random_forest": RandomForestRegressor(featuresCol="features",labelCol="label",predictionCol="prediction",numTrees=80,maxDepth=12,minInstancesPerNode=5,seed=42,subsamplingRate=0.8),
    }
    for name,est in specs.items():
        model=Pipeline(stages=pre+[est]).fit(train)
        model.write().overwrite().save(root+"/"+name)
        results["models"][name]={}
        for split_name,split in [("validation",valid),("test",test)]:
            pred=model.transform(split).select("city_slug","to_epoch","label","prediction")
            results["models"][name][split_name]=metrics(pred)
        test_pred=model.transform(test).select("city_slug","to_epoch","label","prediction")
        city=[]
        for slug in sorted(r[0] for r in test.select("city_slug").distinct().collect()):
            city.append({"city_slug":slug,**metrics(test_pred.filter(F.col("city_slug")==slug))})
        results["models"][name]["test_by_city"]=city
    print("MODEL_RESULTS_JSON_BEGIN"); print(json.dumps(results,sort_keys=True)); print("MODEL_RESULTS_JSON_END")
    print("SIX_CITY_MODEL_TRAINING_SUCCESS")
finally:
    spark.stop()
PY_REMOTE
spark-submit --master local[2] --driver-memory 5g --conf spark.ui.enabled=false --conf spark.sql.shuffle.partitions=24 --conf spark.driver.maxResultSize=1g "$SCRIPT" "$PAYLOAD"
rm -f "$SCRIPT" "$PAYLOAD"
'''.replace("__PAYLOAD__",payload_b64)
r=run_remote(cmd,timeout=10800,show_output=False)
if "SIX_CITY_MODEL_TRAINING_SUCCESS" not in r.stdout:
    print(r.stdout[-16000:]); print(r.stderr[-8000:]); raise RuntimeError("Training success marker missing")
b="MODEL_RESULTS_JSON_BEGIN"; e="MODEL_RESULTS_JSON_END"
MODEL_RESULTS=json.loads(r.stdout.split(b,1)[1].split(e,1)[0].strip())
print(json.dumps(MODEL_RESULTS,indent=2,sort_keys=True))
print("CELL 8: SUCCESS")

RuntimeError: Remote command failed with return code 1

In [9]:
# ============================================================
# CELL 8 DIAGNOSTIC: SHOW THE REAL REMOTE FAILURE
#
# Read-only:
#   - Does not train another model
#   - Does not delete model outputs
#   - Does not modify HDFS
# ============================================================

diagnostic_command = r'''
source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

echo "============================================================"
echo "CELL 8 FAILURE DIAGNOSTIC"
echo "============================================================"

echo
echo "========== TEMPORARY TRAINING FILES =========="

for file_path in \
    /tmp/ghsl_train_payload.json \
    /tmp/ghsl_train_models.py
do
    if [ -f "$file_path" ]; then
        echo "FOUND: $file_path"
        ls -lh "$file_path"
    else
        echo "MISSING: $file_path"
    fi
done

echo
echo "========== MODEL ROOT =========="

MODEL_ROOT="/data/ghsl/modeling/models/six_city_growth_v1"

if hdfs dfs -test -e "$MODEL_ROOT"; then
    echo "MODEL_ROOT_EXISTS: YES"
    hdfs dfs -ls -R "$MODEL_ROOT" | head -200
else
    echo "MODEL_ROOT_EXISTS: NO"
fi

echo
echo "========== LINEAR REGRESSION MODEL =========="

if hdfs dfs -test -e "$MODEL_ROOT/linear_regression"; then
    echo "LINEAR_REGRESSION_EXISTS: YES"
    hdfs dfs -ls "$MODEL_ROOT/linear_regression"
else
    echo "LINEAR_REGRESSION_EXISTS: NO"
fi

echo
echo "========== RANDOM FOREST MODEL =========="

if hdfs dfs -test -e "$MODEL_ROOT/random_forest"; then
    echo "RANDOM_FOREST_EXISTS: YES"
    hdfs dfs -ls "$MODEL_ROOT/random_forest"
else
    echo "RANDOM_FOREST_EXISTS: NO"
fi

echo
echo "========== TRANSITION DATASET =========="

if hdfs dfs -test -e /data/ghsl/modeling/six_city_transitions_v1; then
    echo "TRANSITION_DATASET_EXISTS: YES"
else
    echo "TRANSITION_DATASET_EXISTS: NO"
fi

echo
echo "========== DISK STATUS =========="

df -h /
hdfs dfsadmin -report | sed -n '1,35p'

echo
echo "========== RECENT KERNEL MEMORY EVENTS =========="

dmesg 2>/dev/null \
| grep -Ei "out of memory|oom|killed process" \
| tail -20 \
|| true

echo
echo "CELL_8_DIAGNOSTIC_COMPLETE"
'''

diagnostic_result = run_remote(
    diagnostic_command,
    timeout=300,
    show_output=True,
)

print(
    "Return code:",
    diagnostic_result.returncode,
)

print(
    "CELL 8 DIAGNOSTIC: SUCCESS"
)

CELL 8 FAILURE DIAGNOSTIC

========== TEMPORARY TRAINING FILES ==========
FOUND: /tmp/ghsl_train_payload.json
-rw-rw-r-- 1 ubuntu ubuntu 162 Nov 15 05:41 /tmp/ghsl_train_payload.json
FOUND: /tmp/ghsl_train_models.py
-rw-rw-r-- 1 ubuntu ubuntu 4.3K Nov 15 05:41 /tmp/ghsl_train_models.py

========== MODEL ROOT ==========
MODEL_ROOT_EXISTS: NO

========== LINEAR REGRESSION MODEL ==========
LINEAR_REGRESSION_EXISTS: NO

========== RANDOM FOREST MODEL ==========
RANDOM_FOREST_EXISTS: NO

========== TRANSITION DATASET ==========
TRANSITION_DATASET_EXISTS: YES

========== DISK STATUS ==========
Filesystem      Size  Used Avail Use% Mounted on
/dev/root        77G   24G   53G  31% /
Configured Capacity: 82085269504 (76.45 GB)
Present Capacity: 63990758985 (59.60 GB)
DFS Remaining: 56734658560 (52.84 GB)
DFS Used: 7256100425 (6.76 GB)
DFS Used%: 11.34%
Replicated Blocks:
	Under replicated blocks: 1
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 

In [13]:
# ============================================================
# CELL 8A: RERUN PRESERVED TRAINING SCRIPT WITH VISIBLE LOGS
#
# Purpose:
#   - Reuse the exact script and payload from the failed Cell 8
#   - Display the real Spark/Python exception
#
# Safety:
#   - Confirms the model root is absent before starting
#   - Does not delete transition data
#   - Does not modify the longitudinal dataset
# ============================================================

visible_training_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

SCRIPT="/tmp/ghsl_train_models.py"
PAYLOAD="/tmp/ghsl_train_payload.json"
MODEL_ROOT="/data/ghsl/modeling/models/six_city_growth_v1"

echo "============================================================"
echo "VISIBLE CELL 8 TRAINING RETRY"
echo "============================================================"

if [ ! -f "$SCRIPT" ]; then
    echo "ERROR: Preserved training script is missing."
    exit 20
fi

if [ ! -f "$PAYLOAD" ]; then
    echo "ERROR: Preserved training payload is missing."
    exit 21
fi

if hdfs dfs -test -e "$MODEL_ROOT"; then
    echo "ERROR: Model root already exists:"
    echo "$MODEL_ROOT"
    echo "Nothing was deleted."
    exit 22
fi

echo
echo "Training script: $SCRIPT"
echo "Training payload: $PAYLOAD"
echo "Model root currently exists: NO"

echo
echo "========== TRAINING PAYLOAD =========="
cat "$PAYLOAD"

echo
echo "========== STARTING SPARK TRAINING =========="

spark-submit \
    --master local[2] \
    --driver-memory 5g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=24 \
    --conf spark.driver.maxResultSize=1g \
    "$SCRIPT" \
    "$PAYLOAD"
'''

visible_training_result = subprocess.run(
    [
        str(REMOTE_EXEC),
        visible_training_command,
    ],
    text=True,
    capture_output=True,
    timeout=10800,
)

print(
    "REMOTE RETURN CODE:",
    visible_training_result.returncode,
)

print()
print(
    "================ REMOTE STDOUT ================"
)
print(
    visible_training_result.stdout
)

print()
print(
    "================ REMOTE STDERR ================"
)
print(
    visible_training_result.stderr
)

if visible_training_result.returncode == 0:
    print(
        "VISIBLE TRAINING RETRY: SUCCESS"
    )
else:
    print(
        "VISIBLE TRAINING RETRY: FAILED"
    )
    print(
        "The actual Spark exception is shown above."
    )

REMOTE RETURN CODE: 0

================ REMOTE STDOUT ================
VISIBLE CELL 8 TRAINING RETRY

Training script: /tmp/ghsl_train_models.py
Training payload: /tmp/ghsl_train_payload.json
Model root currently exists: NO

========== TRAINING PAYLOAD ==========
{"input": "/data/ghsl/modeling/six_city_transitions_v1", "model_root": "/data/ghsl/modeling/models/six_city_growth_v1", "target_source": "built_surface_total_m2"}
========== STARTING SPARK TRAINING ==========
MODEL_RESULTS_JSON_BEGIN
{"features": ["grid_1km_col", "grid_1km_row", "grid_min_x", "grid_max_x", "grid_min_y", "grid_max_y", "valid_100m_pixels", "population_coverage_fraction", "population_total", "population_mean_100m", "population_min_100m", "population_max_100m", "zero_population_pixels", "positive_population_pixels", "smod_class", "built_valid_100m_pixels", "built_observed_grid_area_m2", "built_coverage_fraction", "built_surface_nres_m2", "built_surface_residential_m2", "built_surface_fraction", "built_surface_perc

Both trained models were saved under:

Plain Text
/data/ghsl/modeling/models/six_city_growth_v1
Data split
Training: 205,800 rows, outcomes through 2010
Validation: 29,400 rows, 2010 → 2015
Test: 29,400 rows, 2015 → 2020
Results
Model	2020 MAE	2020 RMSE	2020 R²
Persistence baseline	5,503 m²	12,107 m²	0.9890
Linear Regression	6,961 m²	10,978 m²	0.9910
Random Forest	9,027 m²	18,351 m²	0.9748
Interpretation
Linear Regression is the best overall evaluation model by RMSE and R².
The persistence baseline has the lowest 2020 MAE, so the Linear Regression advantage is not universal.
Linear Regression clearly outperformed persistence on validation:
MAE: 5,905 vs. 7,417 m²
RMSE: 10,701 vs. 16,138 m²
Random Forest underperformed both alternatives overall and should not be selected in its current configuration.

The high R² values are partly expected because the target is next-period total built surface, which is strongly related to the current built state. We should treat this as a strong backtest, but not yet as proof that the model predicts new urban growth hotspots well.

In [14]:
# ============================================================
# RESTORE MODEL_RESULTS FROM SUCCESSFUL CELL 8A OUTPUT
# ============================================================

begin_marker = "MODEL_RESULTS_JSON_BEGIN"
end_marker = "MODEL_RESULTS_JSON_END"

stdout_text = (
    visible_training_result.stdout
)

begin_index = stdout_text.find(
    begin_marker
)

end_index = stdout_text.find(
    end_marker
)

if begin_index < 0:
    raise RuntimeError(
        "MODEL_RESULTS_JSON_BEGIN was not found."
    )

if end_index < 0:
    raise RuntimeError(
        "MODEL_RESULTS_JSON_END was not found."
    )

if end_index <= begin_index:
    raise RuntimeError(
        "Model-result markers are in the wrong order."
    )

json_start = (
    begin_index
    + len(begin_marker)
)

model_results_json = (
    stdout_text[
        json_start:
        end_index
    ]
    .strip()
)

MODEL_RESULTS = json.loads(
    model_results_json
)

if (
    MODEL_RESULTS.get(
        "split_counts"
    )
    != {
        "train": 205800,
        "validation": 29400,
        "test": 29400,
    }
):
    raise RuntimeError(
        "Unexpected model split counts: "
        + repr(
            MODEL_RESULTS.get(
                "split_counts"
            )
        )
    )

required_models = {
    "persistence",
    "linear_regression",
    "random_forest",
}

actual_models = set(
    MODEL_RESULTS.get(
        "models",
        {},
    )
)

if (
    actual_models
    != required_models
):
    raise RuntimeError(
        "Unexpected model-result inventory: "
        + repr(
            sorted(actual_models)
        )
    )

print(
    json.dumps(
        MODEL_RESULTS,
        indent=2,
        sort_keys=True,
    )
)

print(
    "MODEL_RESULTS restored: PASS"
)

print(
    "Training rows:",
    MODEL_RESULTS[
        "split_counts"
    ][
        "train"
    ],
)

print(
    "Validation rows:",
    MODEL_RESULTS[
        "split_counts"
    ][
        "validation"
    ],
)

print(
    "Test rows:",
    MODEL_RESULTS[
        "split_counts"
    ][
        "test"
    ],
)

print(
    "Recovered Cell 8 results: SUCCESS"
)

{
  "features": [
    "grid_1km_col",
    "grid_1km_row",
    "grid_min_x",
    "grid_max_x",
    "grid_min_y",
    "grid_max_y",
    "valid_100m_pixels",
    "population_coverage_fraction",
    "population_total",
    "population_mean_100m",
    "population_min_100m",
    "population_max_100m",
    "zero_population_pixels",
    "positive_population_pixels",
    "smod_class",
    "built_valid_100m_pixels",
    "built_observed_grid_area_m2",
    "built_coverage_fraction",
    "built_surface_nres_m2",
    "built_surface_residential_m2",
    "built_surface_fraction",
    "built_surface_percent",
    "built_surface_nres_fraction",
    "built_surface_residential_fraction",
    "nres_share_of_built_surface",
    "built_100m_pixels",
    "nres_100m_pixels",
    "population_per_built_hectare",
    "population_per_residential_built_hectare",
    "grid_resolution_m",
    "from_epoch"
  ],
  "models": {
    "linear_regression": {
      "test": {
        "mae": 6961.287531466823,
        "r2": 0.9

In [12]:
# ============================================================
# CORRECTED PYSPARK ML ENVIRONMENT REPAIR
#
# Fixes:
#   - Python 3.12 removed standard-library distutils
#   - PySpark ML still imports distutils
#   - The previous repair stopped at `which pip`
#
# Actions:
#   - Uses python3 -m pip instead of requiring a `pip` command
#   - Bootstraps pip with ensurepip if needed
#   - Installs/upgrades setuptools in ghsl-sedona-env
#   - Verifies pyspark.ml through both Python and spark-submit
#
# Does not:
#   - Modify HDFS
#   - Delete transition data
#   - Train a model
#   - Create model outputs
# ============================================================

repair_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

echo "============================================================"
echo "CORRECTED PYSPARK ML ENVIRONMENT REPAIR"
echo "============================================================"

echo
echo "========== PYTHON ENVIRONMENT =========="

echo "Python executable:"
which python3

echo
echo "Python version:"
python3 --version

echo
echo "Python prefix:"
python3 -c "import sys; print(sys.prefix)"

echo
echo "Python base prefix:"
python3 -c "import sys; print(sys.base_prefix)"

echo
echo "========== PIP AVAILABILITY =========="

if python3 -m pip --version >/dev/null 2>&1; then
    echo "python3 -m pip: AVAILABLE"
    python3 -m pip --version
else
    echo "python3 -m pip: MISSING"
    echo "Bootstrapping pip with ensurepip..."

    python3 -m ensurepip \
        --upgrade

    echo
    echo "Pip after ensurepip:"
    python3 -m pip --version
fi

echo
echo "========== CURRENT SETUPTOOLS STATUS =========="

python3 - <<'PY_REMOTE'
try:
    import setuptools

    print(
        "setuptools import before repair: PASS"
    )

    print(
        "setuptools version before repair:",
        setuptools.__version__,
    )

except Exception as error:
    print(
        "setuptools import before repair: FAIL"
    )

    print(
        repr(error)
    )


try:
    import distutils

    print(
        "distutils import before repair: PASS"
    )

    print(
        "distutils path:",
        getattr(
            distutils,
            "__file__",
            "unknown",
        ),
    )

except Exception as error:
    print(
        "distutils import before repair: FAIL"
    )

    print(
        repr(error)
    )
PY_REMOTE

echo
echo "========== INSTALLING SETUPTOOLS COMPATIBILITY =========="

python3 -m pip install \
    --upgrade \
    setuptools

echo
echo "========== DIRECT PYTHON IMPORT TEST =========="

export SETUPTOOLS_USE_DISTUTILS=local

python3 - <<'PY_REMOTE'
import sys
import setuptools
import distutils

from pyspark.ml import Pipeline

from pyspark.ml.feature import (
    Imputer,
    VectorAssembler,
)

from pyspark.ml.regression import (
    LinearRegression,
    RandomForestRegressor,
)

from pyspark.ml.evaluation import (
    RegressionEvaluator,
)

print(
    "Python executable:",
    sys.executable,
)

print(
    "setuptools version:",
    setuptools.__version__,
)

print(
    "distutils path:",
    getattr(
        distutils,
        "__file__",
        "unknown",
    ),
)

print(
    "Direct distutils import: PASS"
)

print(
    "Direct pyspark.ml Pipeline import: PASS"
)

print(
    "Direct pyspark.ml feature imports: PASS"
)

print(
    "Direct pyspark.ml regression imports: PASS"
)

print(
    "Direct pyspark.ml evaluator import: PASS"
)

print(
    "DIRECT_PYSPARK_ML_IMPORT_SUCCESS"
)
PY_REMOTE

echo
echo "========== SPARK-SUBMIT IMPORT TEST =========="

TEST_SCRIPT="/tmp/test_pyspark_ml_import.py"

cat > "$TEST_SCRIPT" <<'PY_REMOTE'
import os
import sys
import setuptools
import distutils

from pyspark.ml import Pipeline

from pyspark.ml.feature import (
    Imputer,
    VectorAssembler,
)

from pyspark.ml.regression import (
    LinearRegression,
    RandomForestRegressor,
)

from pyspark.ml.evaluation import (
    RegressionEvaluator,
)

print(
    "Spark-submit Python executable:",
    sys.executable,
)

print(
    "SETUPTOOLS_USE_DISTUTILS:",
    os.environ.get(
        "SETUPTOOLS_USE_DISTUTILS"
    ),
)

print(
    "setuptools version:",
    setuptools.__version__,
)

print(
    "distutils path:",
    getattr(
        distutils,
        "__file__",
        "unknown",
    ),
)

print(
    "SPARK_SUBMIT_PYSPARK_ML_IMPORT_SUCCESS"
)
PY_REMOTE

spark-submit \
    --master local[1] \
    --driver-memory 2g \
    --conf spark.ui.enabled=false \
    "$TEST_SCRIPT"

rm -f \
    "$TEST_SCRIPT"

echo
echo "HDFS modified: NO"
echo "Model training launched: NO"
echo "PYSPARK_ML_ENVIRONMENT_REPAIR_SUCCESS"
'''

repair_result = run_remote(
    repair_command,
    timeout=900,
    show_output=True,
)

if (
    "DIRECT_PYSPARK_ML_IMPORT_SUCCESS"
    not in repair_result.stdout
):
    raise RuntimeError(
        "The direct PySpark ML import test failed."
    )

if (
    "SPARK_SUBMIT_PYSPARK_ML_IMPORT_SUCCESS"
    not in repair_result.stdout
):
    raise RuntimeError(
        "The spark-submit PySpark ML import test failed."
    )

if (
    "PYSPARK_ML_ENVIRONMENT_REPAIR_SUCCESS"
    not in repair_result.stdout
):
    raise RuntimeError(
        "The PySpark ML environment repair "
        "did not produce its final success marker."
    )

print(
    "HDFS modified: NO"
)

print(
    "Model training launched: NO"
)

print(
    "PYSPARK ML REPAIR: SUCCESS"
)

CORRECTED PYSPARK ML ENVIRONMENT REPAIR

========== PYTHON ENVIRONMENT ==========
Python executable:
/home/ubuntu/ghsl-sedona-env/bin/python3

Python version:
Python 3.12.16

Python prefix:
/home/ubuntu/ghsl-sedona-env

Python base prefix:
/home/ubuntu/.local/share/uv/python/cpython-3.12.16-linux-x86_64-gnu

========== PIP AVAILABILITY ==========
python3 -m pip: MISSING
Bootstrapping pip with ensurepip...
Looking in links: /tmp/tmpcs5hxd06
Processing /tmp/tmpcs5hxd06/pip-26.2.2-py3-none-any.whl

Pip after ensurepip:
pip 26.2.2 from /home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pip (python 3.12)

========== CURRENT SETUPTOOLS STATUS ==========
setuptools import before repair: FAIL
ModuleNotFoundError("No module named 'setuptools'")
distutils import before repair: FAIL
ModuleNotFoundError("No module named 'distutils'")

========== INSTALLING SETUPTOOLS COMPATIBILITY ==========
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.3/876.3 kB 17.6 MB/s  0:00:00

========== DIRECT PYT

In [15]:
# ============================================================
# CELL 9: FREEZE METRICS AND TRAINING MANIFEST TO GOOGLE DRIVE
# ============================================================
manifest={
  "created_at_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime()),
  "training_cities":inventory["training_city_slugs"],
  "held_out_city":"jaipur",
  "jaipur_used_in_training":False,
  "longitudinal_output":LONGITUDINAL_OUTPUT,
  "transition_output":TRANSITION_OUTPUT,
  "model_root":MODEL_ROOT,
  "target_source_column":TARGET_SOURCE_COLUMN,
  "label_definition":"next epoch built surface",
  "growth_analysis_field":"built_surface_growth_m2",
  "chronological_split":{"train":"to_epoch <= 2010","validation":"to_epoch == 2015","test":"to_epoch == 2020"},
  "results":MODEL_RESULTS,
}
manifest_path=ARTIFACT_ROOT/"training_manifest.json"
manifest_path.write_text(json.dumps(manifest,indent=2,sort_keys=True)+"\n",encoding="utf-8")
if not manifest_path.is_file() or manifest_path.stat().st_size<=0:
    raise RuntimeError("Training manifest was not written.")
print("Training manifest:",manifest_path)
for model_name,data in MODEL_RESULTS["models"].items():
    print(model_name,"test:",data["test"])
print("Jaipur included in training: NO")
print("NOTEBOOK_12_MODEL_TRAINING_SUCCESS")
print("CELL 9: SUCCESS")

Training manifest: /content/drive/MyDrive/GHSL_Urban_Growth_Model/model_training_v1/training_manifest.json
linear_regression test: {'mae': 6961.287531466823, 'r2': 0.9909752766308704, 'rmse': 10977.947370480828}
persistence test: {'mae': 5503.150680272109, 'r2': 0.9890228158949733, 'rmse': 12107.365945209738}
random_forest test: {'mae': 9027.187055572596, 'r2': 0.9747807145895845, 'rmse': 18351.458394983416}
Jaipur included in training: NO
NOTEBOOK_12_MODEL_TRAINING_SUCCESS
CELL 9: SUCCESS


We now have a reproducible six-city modeling pipeline with:

294,000 validated longitudinal rows
264,600 five-year transition examples
Chronological train, validation, and test splits
Persistence baseline
Linear Regression model
Random Forest model
Overall and city-wise test metrics
Models saved in HDFS
Training manifest saved in Google Drive
Jaipur completely excluded from training
Selected evaluation model

Linear Regression is the preferred evaluation model because it achieved the best overall test RMSE and R2R^2R2:

MAE: 6,961 m²
RMSE: 10,978 m²
R2R^2R2: 0.99098

However, report this nuance honestly:

Persistence achieved the better MAE: 5,503 m²
Linear Regression achieved the better RMSE and R2R^2R2
Random Forest underperformed both

This means Linear Regression handles larger errors better overall, while persistence is more accurate for the typical cell under absolute error.

Saved artifacts

HDFS

Plain Text
/data/ghsl/modeling/six_city_longitudinal_v1
/data/ghsl/modeling/six_city_transitions_v1
/data/ghsl/modeling/models/six_city_growth_v1/linear_regression
/data/ghsl/modeling/models/six_city_growth_v1/random_forest

Google Drive

Plain Text
/content/drive/MyDrive/GHSL_Urban_Growth_Model/model_training_v1/training_manifest.json

## After this notebook

Do not retrain with Jaipur. The next notebook should process Jaipur through exactly the same schema and feature pipeline, load the frozen models from HDFS, and report out-of-city generalization metrics separately.